## Wprowadzenie 

Ta lekcja obejmie: 
- Czym jest wywoływanie funkcji i do czego służy 
- Jak utworzyć wywołanie funkcji za pomocą Azure OpenAI 
- Jak zintegrować wywołanie funkcji z aplikacją 

## Cele nauki 

Po ukończeniu tej lekcji będziesz wiedział jak i zrozumiesz: 

- Cel używania wywoływania funkcji 
- Konfigurację wywołania funkcji z wykorzystaniem usługi Azure Open AI 
- Projektowanie efektywnych wywołań funkcji dla zastosowań twojej aplikacji 


## Zrozumienie wywołań funkcji 

W tej lekcji chcemy zbudować funkcję dla naszego startupu edukacyjnego, która pozwoli użytkownikom korzystać z chatbota do wyszukiwania kursów technicznych. Będziemy rekomendować kursy dopasowane do ich poziomu umiejętności, obecnej roli oraz technologii, którą się interesują. 

Do realizacji tego celu użyjemy połączenia: 
 - `Azure Open AI` do stworzenia doświadczenia czatu dla użytkownika
 - `Microsoft Learn Catalog API` do pomocy użytkownikom w znajdowaniu kursów na podstawie ich zapytań 
 - `Function Calling` aby przejąć zapytanie użytkownika i wysłać je do funkcji, która wykona żądanie do API. 

Aby zacząć, przyjrzyjmy się, dlaczego w ogóle chcielibyśmy używać wywołań funkcji: 


### Dlaczego wywoływanie funkcji 

Jeśli ukończyłeś już jakiś inny rozdział tego kursu, prawdopodobnie rozumiesz moc wykorzystywania Dużych Modeli Językowych (LLM). Mamy nadzieję, że dostrzegasz także niektóre z ich ograniczeń. 

Wywoływanie funkcji to funkcja usługi Azure Open AI, która pozwala przezwyciężyć następujące ograniczenia: 
1) Spójny format odpowiedzi 
2) Możliwość korzystania z danych z innych źródeł aplikacji w kontekście czatu 

Przed wprowadzeniem wywoływania funkcji odpowiedzi z LLM były niestrukturalne i niespójne. Programiści musieli pisać skomplikowany kod walidacyjny, aby mieć pewność, że poradzą sobie z każdą wersją odpowiedzi. 

Użytkownicy nie mogli uzyskać odpowiedzi takich jak „Jaka jest teraz pogoda w Sztokholmie?”. Wynikało to z faktu, że modele były ograniczone do danych, na których zostały wytrenowane. 

Spójrzmy na poniższy przykład ilustrujący ten problem: 

Załóżmy, że chcemy utworzyć bazę danych danych studentów, aby móc im zasugerować właściwy kurs. Poniżej mamy dwa opisy studentów, które są bardzo podobne pod względem zawartych danych.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Chcemy wysłać to do LLM, aby przeanalizować dane. Można to później wykorzystać w naszej aplikacji do wysłania tego do API lub zapisania w bazie danych. 

Stwórzmy dwa identyczne zapytania, w których poinstruujemy LLM, jakie informacje nas interesują: 


Chcemy wysłać to do LLM, aby przeanalizować części ważne dla naszego produktu. W ten sposób możemy stworzyć dwa identyczne podpowiedzi, aby poinstruować LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Po stworzeniu tych dwóch promptów, wyślemy je do LLM, korzystając z `client.responses.create`. Przechowujemy prompt w zmiennej `input` i przypisujemy rolę `user`. Ma to na celu imitację wiadomości od użytkownika wysyłanej do chatbota. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Teraz możemy wysłać oba zapytania do LLM i zbadać otrzymaną odpowiedź. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Chociaż wywołania są takie same, a opisy podobne, możemy otrzymać różne formaty właściwości `Grades`. 

Jeśli uruchomisz powyższą komórkę wielokrotnie, format może być `3.7` lub `3.7 GPA`. 

Dzieje się tak, ponieważ LLM przyjmuje dane nieustrukturyzowane w formie napisanego wywołania i zwraca również dane nieustrukturyzowane. Potrzebujemy mieć sformatowane dane, aby wiedzieć, czego się spodziewać przy ich przechowywaniu lub używaniu.

Korzystając z wywoływania funkcji, możemy zapewnić, że otrzymamy dane w strukturze. Przy wywoływaniu funkcji LLM faktycznie nie wywołuje ani nie uruchamia żadnych funkcji. Zamiast tego tworzymy strukturę, którą LLM ma się kierować w swoich odpowiedziach. Następnie używamy tych sformatowanych odpowiedzi, aby wiedzieć, jaką funkcję uruchomić w naszych aplikacjach.  
 


![Diagram Przepływu Wywoływania Funkcji](../../../../translated_images/pl/Function-Flow.083875364af4f4bb.webp)


Następnie możemy wziąć to, co zostanie zwrócone przez funkcję i odesłać to do LLM. LLM odpowie wtedy w języku naturalnym, aby odpowiedzieć na zapytanie użytkownika. 


### Przypadki użycia wywołań funkcji 

**Wywoływanie narzędzi zewnętrznych** 
Chatboty świetnie sprawdzają się w udzielaniu odpowiedzi na pytania użytkowników. Dzięki wywoływaniu funkcji chatboty mogą korzystać z wiadomości od użytkowników, aby wykonać określone zadania. Na przykład, student może poprosić chatbota o „Wysłanie e-maila do mojego wykładowcy z informacją, że potrzebuję więcej pomocy w tym temacie”. Może to spowodować wywołanie funkcji `send_email(to: string, body: string)`


**Tworzenie zapytań do API lub bazy danych**
Użytkownicy mogą znaleźć informacje, używając języka naturalnego, który jest konwertowany na sformatowane zapytanie lub żądanie API. Przykładem może być nauczyciel, który pyta „Którzy uczniowie ukończyli ostatnie zadanie”, co może wywołać funkcję o nazwie `get_completed(student_name: string, assignment: int, current_status: string)`


**Tworzenie danych strukturalnych**
Użytkownicy mogą wziąć blok tekstu lub plik CSV i użyć modelu LLM do wyodrębnienia z niego ważnych informacji. Na przykład, student może przekształcić artykuł z Wikipedii o umowach pokojowych, aby stworzyć fiszki AI. Można to zrobić, używając funkcji o nazwie `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Tworzenie pierwszego wywołania funkcji 

Proces tworzenia wywołania funkcji obejmuje 3 główne kroki: 
1. Wywołanie Chat Completions API z listą twoich funkcji i wiadomością od użytkownika 
2. Odczytanie odpowiedzi modelu, aby wykonać akcję, np. wywołać funkcję lub API 
3. Ponowne wywołanie Chat Completions API z odpowiedzią z twojej funkcji, aby użyć tych informacji do stworzenia odpowiedzi dla użytkownika. 


![Przebieg wywołania funkcji](../../../../translated_images/pl/LLM-Flow.3285ed8caf4796d7.webp)


### Elementy wywołania funkcji 

#### Dane wejściowe od użytkownika 

Pierwszym krokiem jest utworzenie wiadomości od użytkownika. Można ją przypisać dynamicznie, pobierając wartość z pola tekstowego, lub przypisać ją tutaj. Jeśli to Twój pierwszy raz z API Chat Completions, musimy zdefiniować `role` oraz `content` wiadomości. 

`role` może być ustawione na `system` (tworzenie reguł), `assistant` (model) lub `user` (użytkownik końcowy). Dla wywołania funkcji przypiszemy ją jako `user` oraz podamy przykładowe pytanie. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Tworzenie funkcji. 

Następnie zdefiniujemy funkcję oraz jej parametry. Użyjemy tutaj tylko jednej funkcji o nazwie `search_courses`, ale możesz tworzyć wiele funkcji.

**Ważne** : Funkcje są zawarte w wiadomości systemowej do LLM i będą wliczane do dostępnej liczby tokenów, które masz do dyspozycji. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definicje** 

`name` - Nazwa funkcji, którą chcemy wywołać. 

`description` - To jest opis działania funkcji. Tutaj ważne jest, aby być precyzyjnym i jasnym 

`parameters` - Lista wartości i format, które model ma wygenerować w swojej odpowiedzi 


`type` - Typ danych, w którym będą przechowywane właściwości 

`properties` - Lista konkretnych wartości, które model użyje w swojej odpowiedzi 


`name` - nazwa właściwości, którą model użyje w sformatowanej odpowiedzi 

`type` - Typ danych tej właściwości 

`description` - Opis konkretnej właściwości 


**Opcjonalne**

`required` - wymagana właściwość, aby wywołanie funkcji zostało wykonane 


### Wywoływanie funkcji 
Po zdefiniowaniu funkcji musimy teraz uwzględnić ją w wywołaniu API Chat Completion. Robimy to, dodając `functions` do żądania. W tym przypadku `functions=functions`. 

Istnieje również opcja ustawienia `function_call` na `auto`. Oznacza to, że pozwolimy LLM zdecydować, która funkcja powinna zostać wywołana na podstawie wiadomości użytkownika, zamiast przydzielać ją samodzielnie.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Teraz spójrzmy na odpowiedź i zobaczmy, jak jest sformatowana: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Widać, że wywołano nazwę funkcji i na podstawie wiadomości użytkownika model językowy był w stanie znaleźć dane pasujące do argumentów funkcji. 


## 3.Integrowanie wywołań funkcji z aplikacją. 


Po przetestowaniu sformatowanej odpowiedzi z LLM, możemy teraz zintegrować to z aplikacją. 

### Zarządzanie przepływem 

Aby to zintegrować z naszą aplikacją, wykonajmy następujące kroki: 

Po pierwsze, wykonajmy wywołanie do usług Open AI i zapiszmy wiadomość w zmiennej o nazwie `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Teraz zdefiniujemy funkcję, która wywoła API Microsoft Learn, aby uzyskać listę kursów: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Jako najlepszą praktykę, najpierw sprawdzimy, czy model chce wywołać funkcję. Następnie stworzymy jedną z dostępnych funkcji i dopasujemy ją do funkcji, która jest wywoływana. 
Następnie weźmiemy argumenty funkcji i odwzorujemy je na argumenty pochodzące z LLM.

Na koniec dołączymy komunikat wywołania funkcji oraz wartości zwrócone przez komunikat `search_courses`. Dzięki temu LLM otrzyma wszystkie informacje potrzebne, aby
odpowiedzieć użytkownikowi za pomocą języka naturalnego. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Teraz wyślemy zaktualizowaną wiadomość do LLM, abyśmy mogli otrzymać odpowiedź w języku naturalnym zamiast odpowiedzi w formacie JSON API. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Wyzwanie kodowania 

Świetna robota! Aby kontynuować naukę wywoływania funkcji Azure Open AI, możesz zbudować: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Więcej parametrów funkcji, które mogą pomóc uczącym się znaleźć więcej kursów. Dostępne parametry API znajdziesz tutaj: 
 - Utwórz kolejne wywołanie funkcji, które pobiera więcej informacji od uczącego się, na przykład ich język ojczysty 
 - Utwórz obsługę błędów na wypadek, gdy wywołanie funkcji i/lub wywołanie API nie zwróci żadnych odpowiednich kursów 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
